# TP 3 — Les méthodes de Taylor d'ordre supérieur

**L3 Mathématiques · Équations différentielles et optimisation · Université d'Artois · 2026–2027**

Au TP 2, la méthode d'Euler (ordre 1) demandait environ $54\,000$ pas pour atteindre une erreur de $10^{-4}$ sur
l'exemple fil rouge, contre $280$ pour la méthode d'Euler modifiée (ordre 2). **Plus l'ordre est élevé, moins il faut
de pas.** Peut-on construire des méthodes d'ordre 3, 4, … de façon systématique ? Une première réponse vient de la
**formule de Taylor**. Dans ce TP, vous allez :

- placer Euler et Euler modifiée dans un cadre commun : les **méthodes à un pas** et leur **erreur de consistance** ;
- calculer les **dérivées totales** $f', f'', \dots$ d'un second membre $f(t, y)$, à la main puis avec SymPy ;
- programmer la **méthode de Taylor d'ordre $n$**, pour $n$ quelconque ;
- vérifier que son erreur de consistance est en $O(h^n)$ et qu'elle est **d'ordre $n$** ;
- l'appliquer à un problème de physique : un **projectile freiné par l'air** ;
- ★ interpoler entre les points de la grille (Hermite) et étudier la stabilité.

> Comme au TP 2, on n'utilise `solve_ivp` que pour obtenir une solution de **référence**.
>
> 📚 Pour aller plus loin : R. L. Burden et J. D. Faires, *Numerical Analysis*, chapitre 5.

| Partie | Contenu | Durée indicative |
|:--|:--|:--|
| 0 | Mise en route et rappels | 5 min |
| 1 | Dérivées totales et deux pas à la main | 15 min |
| 2 | Programmer la méthode de Taylor | 15 min |
| 3 | Erreur de consistance et ordre | 15 min |
| 4 | Calculer les dérivées avec SymPy | 15 min |
| 5 | Application : un projectile freiné par l'air | 15 min |
| 6 ★ | Entre les points de la grille : l'interpolation d'Hermite | pour aller plus loin |
| 7 ★ | Stabilité | pour aller plus loin |

**Légende.** 📖 à lire et exécuter · ✏️ exercice · ✅ vérification automatique · ★ pour aller plus loin

## 0. Mise en route

1. **Exécutez d'abord la cellule de vérification** ci-dessous (elle est repliée : ne la modifiez pas). Elle définit
   la fonction `verifier(...)` ✅ qui vous indique, après chaque exercice, si votre réponse est correcte.
2. Exécutez les cellules **dans l'ordre** (Maj + Entrée). En cas de comportement étrange :
   **Kernel → Restart Kernel and Run All Cells**.
3. **Sauvegardez votre travail.** Dans la version en ligne, le notebook est enregistré *dans le navigateur de cet
   ordinateur*. Avant de partir : **File → Download**.

In [ ]:
#@title ⚙️ Cellule de vérification — exécutez-la (Maj + Entrée) sans la modifier
# Elle définit la fonction verifier(...) utilisée tout au long du TP.
import numpy  # import visible : JupyterLite charge les paquets d'après les « import » des cellules
import base64 as _b64, zlib as _zlib
exec(_zlib.decompress(_b64.b64decode(
    "eNrtfV9vG1mW37s+xe1qYFXVTdIkRUk2bTnxdGtmvOP2GLangYGiZpfIolgWWWRXFSnRai0mwe5iAwTYlwDZhw0y"
    "WASb9QbIw2Qesti31gfId/AXyFfI75z7p279o6RuO8jDGrZJVt26f8499/w/pz4Vzc+aYjgfhdFpXyzTcfM+Xdly"
    "HGfr6+t3cTgOh34azqNE+Mt0PsP375ZBIkZL8fqF2BHvf/cfxTP8nl2/SyfzEd0JxGt/PZ3HYrQ9j0dxIJLlgnoK"
    "lnFra+vrIE7QnXj29NXTnz07bIj4+l0SxKvrd4G4/r2YbgdREoSnkR+lLfQskmEcLlLx7bcny3A6GqSLndZi/e23"
    "mPJsEQdJEmwNA4FJTsIgFkGKDsIouf4HDDvyMempL4bBdLqcBmLoDyc0CqYezdPgZD4/E5j1chTSWFtbv0n8U/1U"
    "kDXpC7EKCA5B7DrBxbznNETKC/S2tl7Ml7EYTnzARIzn0ZAgRctYxPPT2J/NMCd3MQkH8gH9YANAisNVECSDdJ76"
    "0yDxGpjp1soGOK/QxzowGYLRcpr6KfW9jAIRzhbT63ezIEpl2xG3GeP5IBoSxOOtxXSZEMwTLDwRaZCkSYMAFAfD"
    "eRT51/8DsMJ2BXEsG039JJF7u6BFjdAIs19GIoxG4RBgDk8wJMAxolnyfmF6aQjoBsANMcVgNA9APGshl9fAWoIx"
    "FhZiwmLyzZm4J84+2bIektfcs8873icNMfaHKSYlJiKIRBrPFzTxIbbo1Xr2Yi2ibSyGQDCPU8zou+U2I04YpQrN"
    "BOHhKo+93y1D6m2OCZwEyTyMttzgIoiHWFoi9gkw9z0sDj0BfrM5gbY5vf499mXbIAm+8+LeoJUZnhEGe7Kl8WwV"
    "JgAVLTrgUZMlls4796fLxToN4mdhGkggA3PiU0B5SrAE7IGGdPBk32KSzqYCezegL/oiTuCEL9KXLX01Ws4Wa74c"
    "Lba2tj4VzQ/2B509ob2b8Omgo45Th+/Jhx1lFIzFwOeBcNAS4PUyxXkJ0zig45L64TQ5cByvvyXwB2dthLvBRYq7"
    "IRA6EAfikm/RH2d+5vSF63wa7I17gY8T63zaHu3tdkf09f3f/oXjNbLGZ3PZeDwM7gd73NjfbQftgBv/5/+Qa7wK"
    "R4FqHoz3gzY33/d7J+227Puv/8//+mv9xNWRXMgx/0rjdd90NI7nM/H0xRpUM2qNwmQx9dcKqcQvX3/1DIuWF7fM"
    "I+qCS7ddc5U72340ClciSdfT4MA58Ydnp/F8GY36lwSoq4fDOahO/5LhdfXwBGQ5iJvTYJz2dxcXIplPw5HQd7cL"
    "PS/8ETOH+2jZ6S4uHs6As2HU7+F3W/cV+6NwmdC1hxgxbY79WThd9xOcjWZC1POh87jY8aOTx5e8dVcYm/b56tG9"
    "k8fb4nPhjp1HJ/HjS7XtV44IxxoHRDBNAgFEQDvn0T0s+7FjOvY8/hpcDAPwjEP+wOnPoK4AHAcZVBcxCAdGLMzF"
    "8bKHzOD93BLkkw5hBSYTB61keeLGzqOjbx4ff/6YsMExqIuZbW0xjRWDQ6D40DWzUyiNg//y+t0CtArkPRrOQZdB"
    "BUGSQBzUkRNE9+QZAeFhopcxMCIcW/IYBRchqIqL9YxCGqGhO1BDYUFgb8Lcz5YV+yGGVzPUD+luo/nsJA7ciwae"
    "nmVdXYgwEYfTabhIwqSmK2woZIzg8SWexDbzd7GCRAGajKUGQt5utVr6Zp/5HzhOGrwlENDVltoUPehzbNidB5TX"
    "6dFsKPAsw70B4ybzND0++FhD+KtgSMxQPh0H6TKO9PP/Sk0rd75lEzGezv3UvbgJL2+e+yhYoRHYyD8CO2kmcjew"
    "tXhUDJkl4ur85A1YGThqul7gmOTwNTt7egBqhMm1BoPInwWDgR6NAK02PfXBy/zlv+z6XXYdXLjlJ34cg1oDcCOC"
    "8wGjwodHBEhcYQJCMV/STNV2EQZIBEmsvYRUin5dvyFOIHen8+lBJ2juYZnq6wO1vVVLgiw8dXld0+lwOk9y3dB/"
    "qhv670YyrPr8uQ9aric3ntE5kW304XEuL/qtvdOrjLKxtDogSLhBsbECFotLrSAZ+kBuxvAgw3AQ6u2+ICaTpDFu"
    "eBquZgh/sQimrkYMxvqG+Aw8b0kyYZJR6yfUcCq3wOARCcCWwPgQsAQPHM/jWZATtyGKaqI+lfIiE/BaMqL6d62J"
    "2DB+jWUexjGULsiA1unAmXECdUjBRSO16jwfK+Dc0ylGJYwiYa90QlncXc0J98yiK7DUPsiQNJKWxU9vxnEg6mpO"
    "OhXBVoIMPV4Wtl/zaAUD2WHhKFUiYgFIH3I+H1z2fgFVBQQdanEKDS/4sN0Pvj58+fTnr0h2vjJHLIqDU1CUGBJE"
    "RvHp1giaY+yTZmaOh4VJqq8jPHOMDnWLOky2j27WsZqEUbjl4YsDFouss6dME6xtxlpqkuYDpdTBYoCHv/3WHCsW"
    "emYs+OAcqNla0ze6B+kD0EGdQ90TSWNRtKQdRw+fxFdOo4KvmvYJC+rziA51godIOCRhsPUGW+kmkD+DkavG9zyv"
    "ACA9Vz9au7HNYwG5WMQ09wI4CrNnBYXmzxiM8ReE7ZLfSjMBP129hpcBDq0/ZOZbcfih8KuzT7+hlV9cvxsumVcb"
    "O0ujUupwFkssIQ6mfkS9D4MUBCanoucoRAaKHDXUJPMgh21uhiBlelA46+VtrgCTEr5xWzGJ21CR+p4ziFUREjQt"
    "0BE1HHhjZe/Qbe3eT2J/NW8oCVDNXHziGJlfAG2cj0KZ4jksQv8wI/MbWMZ8ibHcny2hD0biT8DbQ3CRhvjhv4nd"
    "1g6hCqFWIDoe2Vpy5kLLcvURbArjwcnYhSlhnRcX1qIpUvHZZ6KL49kxQkjnNq3xoys+E6l5qnvjU+oJfNJYgy8H"
    "P/s5sPiIZ9dQ4zZUT/rzGD19KvBzvE3/5H/b0I7p/wP6YSS7NU8gP7qb0lw9PX67tYsZkAQXXCzQVj96GoeQYty0"
    "3RCvG+J5vg9qPoVBdQGaYJpwtw3hvqbltT0Y7p4bCRO2zjgYu9q+CRtNQ5w3xKQBMTwmkSg4ICE8o+Svv3Ejj2B3"
    "7mFNyRw2UxY8bIugGH/jnulGD0HZdV+wA+Pox/PRMkxt3myMmYb6R+h8GkRmYkal0F2JgwPh4InB2MlOHT3V4V9y"
    "YgeAYlsZoGJxxsTYj04D12aExU7Jmjkge1T1n0+NQVQIV81/jCOBuaAHL0dLh5jBhDaUHmALYIt6n2O0KSBk2gbT"
    "4iQmg7POxikwpDOICxfncTUHh52I17Z593XNjNwziW0V8+IbG+ZGVqIBta+aIIOnBggb+pRAH1Qv214v7NB3Xc9m"
    "OMP2PuhWg/pT4U7udb1vyKnww38X+C5++Gdxev0uAvnzoQdIqw7GSUuTcieYStcrrtvmETdjSH4nJE5/foCHPjMe"
    "iaOzY3nQbDLALYluPX3+5dMvDl8NXvzyqTG4ZgjuPAtyln6gy/gRvD+PAbNH9+gLW7AmfO1MXbknYAhUU/uELIDs"
    "pYjAX4vtcEL6JeHCYUP6GR3UhiAFbA5SAJ/BEjIkQ9jVvXkt8ZRdGehFSjaV4DEmECkg6ZOj1ILMmWMUIhqQD8ua"
    "LPk+LWdBR5fWIl7L9Ud6/SDbZOAkNAABx9f3v/v7osTEB4KsH5kf5HQ6P/GnwpVcfkpuMZgAZz47GGZwDYULHFCW"
    "zyZY3jk6nhTHxvoL4/y47dJ7ZSBkn1/SHmd+RCIOmVckbFlN7kt7zk8fLn+0f+IihKuEUJJhoTws5okve5lUW8+c"
    "R+HjIH10L3xcWKBnYYymAT91cn1CauAK/vvhD/f26ON/3uv2gDPlyTkKzxm37A7NvBR3E86rgPxUmD48qLOQnaU5"
    "A4avTEpsU0xDJkyBNJgpBk04T5Z0YDzWSiifzqEHJIzzhak5pEro/lngk0dJUxx9kFw6xBAx+EREJClJ+xsxYflE"
    "joGrxxjsV5m1ko5mUQSRcssaH89r5ZCvlERa4b/ODfsj5Q+Q1AmpLUVpq1IMmeA0LS8sOYQetcQt93lGyM+pV0hp"
    "b4N4nuRuHLVJD18reSUkMD5nqBaHO5kvh9MA2scSG+pIB8tzfmpFAh2eM83tR0mLtvmBfA5ANQJSmAlIGN8SkVLq"
    "ND0KabbHpfkw9RzgCTUXNDzO3DUBN6sUNUPwLbSV4iY1yHjduR7sgJsQDZY9VQojEzUyOKkcMMcNiT3azPD1k98+"
    "+/XLjB9m8ycK4Ksp85mRRtvrdyt/uqQzhYNDxtrrdyfLlM8OghNIEe1XEZ8SC7LXzutWqy/xMLWoPHnO+Ivsd5LZ"
    "7uSFMswYHncfXuMz00NanvTg0zfM4gRURw9ZQPTnhjZAw4R/GwiMh1hAYEwHDQrJFGpGyqOyBD9mGUkiB6jTcrPR"
    "h/50uJwa4iaftndKzqqAy8/tBV4V8aC1XIxgy3IvV2q9Rk1eaR5QJ0qYiASxQ8waK5xt1YjsfL5WUPKLx7CF4IJZ"
    "4npXH0HjNxaxDsf9mDCPRMV5JNL8rVV9Yxb4wEbLF79++vw1GS2PXKhjDdJsSR1tk5mh2Wl16UenBc7ZbbXl931q"
    "1KPvXXoABgnvWLMMiRRJ3uKvaJXxpMLpQY6VzO7ZENXOmXnOM2NzVWkf0FYu21HEbqIj7V6tdkDww8bs416mVw1x"
    "ub7y4AcnTKDbjAkSNsdGu1dBRPFAnVV3fpIG0VL16sMKF43wS7kljPI/VlxFzU21k0uoGVAxMu1k0uOgL69fZdij"
    "XkAWEMGVmarQoZpJTp0u9mnNTPawaWKbnR2VrotLNR/tYOA+D3SXR+T1l1M4nfkX7p/R9zCRTjF72Z4n2Vb9kNk+"
    "5pyVl+wMk10dqVFbCMMKLly5Uu/Yu7KV8gpmAUzhXnJb5121xEvGL6yS0JJPsCZIp9wIpiHPtvfGaE8xE6SYVtp2"
    "HYSKDcUpi5zvf/d3RtF5/1f/7vQe/iMz1Nj8Wkt/5ODFk5evnx4+e3ZIFN9ZzZcJSXtvDUnOx5GRTqWDzZhPFmfW"
    "El+RH3xNz4DfjtgFjkWCBI0JpOitpDnmupcUTPLge6NUHGXQOC4sZr2drWbzOgevnjx/Nfg5LzGs4rzkVFmRwoEn"
    "ZH/9G4fHZNfbuMFzNcMVtsZh0Znbkc6ZEZ/BV09+8/WTp3pWOkiyHLIH9MgUy2y9BHieAXXLU3Ah+r+Zk1MTpkrm"
    "6XZ3ZX0dtq3h9bsRi1USGf51zulEQZYdXJfUmTxBA7rijkHRx1CFxjvqSK86dCgN/cZtZ9xRR7ZM+FbqfmZtPTJT"
    "c6f+7GTk80HviyabatHO4KgVAFbVlEQRNJebXdvWmI3RvI3PdeUjyiJsnQhpW8CJGAtC/+bs+h9ngYnDJPeqL1zs"
    "zVgiyAEmhQH1BcLQDsFZjqEI9aqbA12XQNOtB526nxmoN4OOFnoX8DVvA76cMb1nPWcQOg/GTj0cty3CoKAY01UQ"
    "kgRoD3yOSwDbyQFshwCyUw8wdf//P4DtVADsOCeSuM6YzjY//cMftLsjJZl7rP0OdgO+iQm1RC5eXOrubDGYLmWI"
    "zYUvA9oKVij0Z2gdYW/AnCSVfOQeIep6m5pAJH7NRgb2TVrCZ7IMiYbSNwrvvX53yrIodpMmDNNLy/mYwnBXCsPB"
    "8oKpX9l+0CVjLzmnfQrWimDz/bBzKZPPbol8dt1zoMx5V5HOcyadSuKkO865ppznXfsWnfzzbhaB5Z8kaM/uJMRL"
    "PxIUNVQTSaEEnfOOlibIe3KfIkO0ETMLa5dWp5mxwVBw6CGMVDFxGNhtbd4Jo9OsNrTN0aZVNwUanTxuExqRTfnc"
    "/uk1DHoWGR/RAa3FKGWnVb38vR+1/r1qQ+g2G9uwakxfGsEJlyat2nXK9XTkekhDt9fHqvrReDMMLDv05obHGgBa"
    "HbKhsKOhkI8nGJdWb0m2552cIPoBllIgKbdemcKDCe9Nt7jRXaZr3Z3ubbe6m221fI7iCsg/QVYFEIiQ7QBswiAK"
    "xYYhUK1ASb2EDHCe16J2agOqkYebJ1eSFmCJVRl/CiVT8KWq7VRr7ezev2FDu1Ub2q3Y0G5hQztVG1q3mJs2tOY5"
    "vaEVYCho3m7pBBEyE6iKs5dQaai0GMoskVKAYmgJPdh9wI9S016nOHmXKRln1FDgJhFBbtvZ7XoflzXJ/Cw7znWE"
    "NJlhLFOHLFPeh57BM/IfMC/MDP8g31MEUnGKkToWlJUEyNgZUpDLMvsc/KJDUsURSkKPo2NKbpoz1aQoWF5FSRoY"
    "0jOQw3B5GyGxUl4et7YGvyCbEe9NTlAi/X0I4zlUJSJp+yybl5t1Wjt0h/9B9GlUd0TBFc2UiGu31anpKeU+Pieb"
    "VGPreGvwxRPtUT2CHegXMNwfk8EKslqHjm+7xeEWuNHvHtM1mLCaMHrRne6uvrXDz+yz1UuZu3JoiEb8WI9u3uf+"
    "+VEKRZEda2Ma92vuqUnstDr7Dx7s9dry7nGl2rZTkjt2rPS3OqOa1aJxg5+1bFrTx5psP6WoEzYEKejmozrKsSAs"
    "5UsDcSb/KGOcjaBO9gPyPTnKso7MwF6j1nALSlppQpemoHP6b0J2PctmNq7xd6ixbFMZ0XK1iiYbocSjA6bmQDky"
    "VjHyUCM2UBViByA9h9Ey2MrZmJVzpKHyAOvszfmuijO5TWCQd5e5WmksbkUID8InR7TPpPv2K7eiYJR7oj2Nl9GV"
    "RbNgC5TLvqqPX65+tMG8ljeVXHS8sSRmYHONt6OM5DropjZxQ/JbCde8DVDeIVDZrLhOlrgxLsFlj7PHKhRHKRDf"
    "5oeiZkfpZtEn2p2Na572NhfY7BOZQwG6zV7nzK5d8mYdjY8zpM4kqbEK/GKLJbrl2I6yQc9RbraiJiGZr1vQyTof"
    "mfP2Spz3YzBapm3G/ShZW4midw1Z7+CLsy5o97D0SS2VIvtwXtra8qHJf6GT3oY+erKPXtaFYkx8lHcU4wKPwl8n"
    "S10ZkaSSHTqSCjis4Agd09PU536uz86xXpJiV3Dq5LvcLqOckIKYJ/vWj6v+d6n/aq7WK3G1nruZoxW52U/hZLng"
    "BYifnA9Pq9G8Te5/3wrrIkM5sQzJvAzjqmda1hgZmdOrUv3Buk7Ulhz8NevCWjT9QkgMOXVJVQ8jaaJkw7GVR8Un"
    "3co3Kw6LrJgI2bMIClYzgECSLhdkmqb5g1sQjSc+Lm97RO27Zb5bvwVZ0Bbp49S1mqA1M+GyYUelVV149kxTSdpN"
    "ipycB6Q3gjXQqXSHsNY5t3sgnoheBorH3xi+kuP1aSuZIMeJ1s09qJ8ElvPsVlq4pZ1pqRrXy10F7eUHNnnNnK+N"
    "TTWMcCYobFE6fy1IYfcVfN9WQpX8DsgCkITDq0KAtAXOO2MezlPEcsO3vCTp+y8sRk67vP9jh4s0XFpH50oznzSH"
    "DyT/BFEY82LkGODX/OVKSGRMkwqO6khtMzQaY1owYIRkyCnEHtYGVm84EucFiJznIXK3tZ9Xr724bq3sUgRGEEkZ"
    "kbSuSzn4VeXWGWziw7Bu33Fb6LEqKwMu52WedQ7QmPS6fdXKZ24TzbKQ+7wCuW8Uc1VgR6WkW3BRnzfuFoRmhF8Q"
    "h9tJqmWwVUup9Ici6jZ5qzVAarJBPW/rThPJ9u/y7Kp6B8+Oq6RWnoS859QrTo47tawvth9RWf7fLMnq4N0s/lp8"
    "4acHORXk3a+rciPz3AcVFABDZQKBS0NJT5sDHUvRuyxYy/aQnFUrnQhUJQPLmj3djyv47rLgq+IgR8RYIySvMR//"
    "wLLvL3Euf5OPU2krcwkLrSwYtrvyo9PdPbZy2pd8OhPUGQoOOAtZZz4DNe14ShcSrRyIqZ2XSWhRFg+G4Xats0t3"
    "3yjjAwI4UAOIwsFkL4UjjuGOIunNe0MY6MpEngkZouRXTN2jSLiJrUQrCT/K2Wwm+RhFLARkiVcoYxkZTFCmcafa"
    "575bEnN3qXFDx1kUooM0ED0bcFrg4ecgcC4zizp+ZEIJiSSvLaEku/mJvMk/bkjRxUP6RPlS1SLH0KXp68py85Kg"
    "De/KlOBhAkiyY9dRYcc9sszWKt+qE6qmElndsKemQXbJ1JqGWcVV0d5eggT/qDa3W2vMqQ12on9pQOH25NrgBP1P"
    "ILEZAl/RZL1iGM9Yx45bsLN8IUbIok0tHJ7X8TJQorjNaSXe5EO/ciLkCwiHMm1U2YiAmnAUZ/JR8WgQ6iITtUh2"
    "Nx/FAv83fR7z5LLviuPtZsyvs1O2aFV1o09V1p1HWXYM8dt1XIHez/jUapMSJZCWoqMm4n//uUspLc+2K2ktuxNR"
    "HEeG7uu8kA3skFkK90l0aC1p0NrNqE/BYuS2G/J2y9koJTyjZAJ7OSb+1UJtrudlat7ckn9bky0K3Gryucu1Cyl4"
    "CEuPNTbOws+4sCnqNo6JOJvFpQWFwHLBxUrzI/TpN0S/CUHknv7V6av4QZuyamKMWemv2UmNLbpSo/bBENChIfJi"
    "X6/2oFJxPz2QlLK0zkfWUeW3fCuxEf3S5z1yHrAY0rz+fXNEzps6M2R21rPVq19YfeGsa/p5u1WWVlhNYPXaKqgs"
    "xRqUqezAorPkDSysvBzfcqc1crA6wIstz2oGyDoo3TEKYRG9u2B6N1BYgg7zEujY+VJ3YjauT4K6vHgl3v/7vyJr"
    "H0QbuIE6exiqKw27yoIrviS6QelzTDi6ko4EWKH8bTcuL3daQ5CKhtlSmAwX4REnZDnC31+7E3ucj2yu3WOpVc5D"
    "5tp9YGH1+QAFL19TsOsRWWG7+EeutPtt2gGyjXbbx/kiN8lATiQwKudhSTQlPUF37d1FOH1eEE51HwXx1Bi4jEJb"
    "FD0zu/DzPBs4zHFvmjZpn5pd6oR1NDn3cuh7mEWuL2NUanR9XRRIXSAQHpuVSt6CxfgXrdMgHcCmEuR09Fz9CKub"
    "FrGYaOS6dq0m7o37uUAOh+96+dpNDVHdeF3V2FKe6wshldxuCgpqkpVy+l5JTt9zD5UFHmzBv8jL6WVsypBJMxU8"
    "7hxmbOSwXlA/vLuYflglpB9mIvom0bwhamTu5416MZ0F8Q3y92GV9C1K07U4QZWQbcJqzFp+lEx6eGuJtFeWSCPx"
    "uGAILXfXvWV3t5HdjPXPLmXF0XJyvy558CsT9bRtttHE4QfJTVLo2ImyxNTE9qhuErN+tBh6aLkvzPpkalq2RtCu"
    "cObfMHkHrXKW4O+1LBnqsKamii5SF76/BVfUC1sMsrSc6fy069bJjKq5fb41bXDkl+ykL7KjsKgRpVB8t7m4i7So"
    "NE+WFSMKxk3DcU5WzAzgtIxDaxGHegk1kt/tpru4neQn53lbuW9RlPsweTtQzD2UYuAhS4G1YuCPWfUEHCdFISIf"
    "pQQdw+QcqXWXbl6g8N00cAqLdl4gI5lz3GWFSDkb4Ktavkw2spx04/CUeAmwaDFNqbLpYjpPwTxq5qjYrxzcZV+c"
    "g4XKQAx1c1282SiVI0DRbBi5KNA3CwKDMAhL74SLCOCpJv4R1+C8dINNGALX8bfKSDBRRwfCClXFMPZCI/YU5Atb"
    "9rgFRVcRCQdcs+siQ9FJDYrSBMgjiHKHUqLgH9y6YHE3f+zH10fKnK/6yLGO3D3ZZS3936pNLL1QSXQKDhW+OV4y"
    "Mc0/DYhwKhBwZGtiMooNejW4Phh3xmengnVoulxPX50gyiVUqozhATv866zxv5bEaH4ia8wn5M/V6cBKpvBKNdqq"
    "9K1FSd/Sf/AkzDC3VW0gy5irLM30tbF8OqfjUaX7lJOo3v+Xf9KPKbnu/73OtF+XBMwmES7V/hHKeSXr2cncUopU"
    "RWdclgXQk4Us7XwKfJKgQVOuvROws1VVRk1yrwGQoUXX7xRRYdavByqU9JTWbqISp0TUXIfsL8RKrXgF6w6Ow6L1"
    "irvypB3e/HYpSIC7W1vdrWu7W9/U3TqP/AlYIdmJ1olWqYpvHWCNjjSIBewlUXVpjKwXIowZ9GVUK8faQCGTnRwb"
    "yMvfhnYOrEpZRKY2VMsyaZ5OXjylXrHWUTgeu6c0pZsrSt3YA1WBzS6sN3apMjYHp7fp9RQ24NqeiyWbKnuQAKzv"
    "hiGvddjT3L7zLbs+xavffvXit1Z5igzCKj3OSGabc24lwc1P1Tgnn2aJtvXJnywH5XJt+5W91gOgsrIFF7UJ6ytb"
    "yL6sAsqUcVsaN+sddZYgB8kUVy9Xd0hhAARpOzt2U1eWRJWbhZs3kK63q+3MDklAWR+nucoT6ljPAlQ2pr4p7B01"
    "gmVh5uzgZkVuvlCvFOFIneyJRFJrYiEUpK9eAkKVNVAlBa8RITsiqoqSwss/k+t/jqkl3Iso9BCOKbO4qnjx2Jeo"
    "zcHvaIbsdTkhiCENUpfSiaW1jU9qW5+UWytkR8Qd56/IOutjX+W5g8ioKyfqSm1wcbFdpVhkVxY4kpUmEMdoCk00"
    "VXUJDqzvtB54x3esgF228uyXrDz7bumtMTXhhxVvl3EKurO6c8uQxI1MgAs5aCYiAYR2TWKSumAmjJ0EHrpCfdBl"
    "mDzpEvYbjE7foYNP4/CtNb501WOfZ491vON+ngxmoY7F5WHdxUtUO1VPtizXyv6qQh3rIHfHoMcqYbUm6pGbYvn0"
    "CfBTfKLn1cY3btxZE+lobG2Jqk832i6Rgapp0TG70HDJVSRm4Zj7q5rbM11KzFhWblf5X9fiqZoLGb3lgMSYo9rQ"
    "MtL4LuVeX7GtTmoYjay+WT7yTWsgNkCcis5dVZGVAwZU/LsyfOlKeygSxupqLmAum/dVq5zesVk283IG/JrCo9pN"
    "XeQHPCiiq6SeSJ8l3rAxDeRWMXIsZGwOkauf160lU+/m6ddFqtnoIN/qpN6gJRCsVh9CV9dV1csGqFq0HCF7r0B5"
    "KLvCnrMpTcjVyHWWIVbOU07e/6zvtk3F7VoRsIhR8W31mrJNBsysMxVDVsqOpvpkWeDdTcFvSrDM1TCmVBK2Dkfb"
    "Unfj1L4pYvXpfTQoBUSJV2RBJklDvqqLQGbqNBTMalZe/ScfVd29z+ouC4UdYlOC+RaVHSoXvTa546g01ep0vY+g"
    "B8dnPXlICgHi/ZvLDd62ZGCx6ll21s46Mi/HBERmZ+asa25xLOU94tnn1i8w+jOrCO3Zzi3ad632vVz7hh21ebaz"
    "qeAf9bYnqLIv7Z/sV3/ZwZezXlWVv7Jkdr8kmd3nOgnncMge4vOwdxvVWek3CpkSqejYohC3+pKU62qZGGqGw699"
    "UyXATpkcbySkPVWQKmVO0C24dr8s5wyxp1oZevmZXvGZG9p3ymN06sf4NCsWUnihYV+8/FXPvI6HdKJuu41lg6wx"
    "nVgEeGsXjTgNmFpRAkI6CPVMUIuRpUR9aL7kuPTCLHrtdluL7Vyp5ICfO+r3MVbmcebCaee8Op32xDIvlcmgChsS"
    "tkp0dc7pdZGEGQQ83oHsFOWSR85l3bmKmP+bQgVzb3QqFaMrBfZ3O1lMv6x70FEJUZjzsZe5NVWAfyGe3gp0p/29"
    "c3U1xViyjc6V+ih6Di/544qrYSTqrRQbeaZ64FYOxKO+an28QSDPr9e7JeitApd+DqkLhmE9XcZsSkPrtluVQmch"
    "e6QQm56tpEEVdLvZuSqsLOhaXkQ7IIPpQVNhvn4DRa++ca/Y2BwNei2IndYkD8chHQ6ijnIG8pDwCTmkE3KIsyGH"
    "kyfFPiaBlY8dFE6JyjMOMBuuYsFGevKq6dnRD09VtNhTqr+aHjlBuqRxslWAPCz/pgR43XkgM6UeU1uqAsC/fxTq"
    "6+goVYyjK3MUWebqE9ZG8sVnOBTsc17O+FgkMlNgs/uZDkc+FQkI933O4Uxhkfwb01d+y7yH+vtK7S8PBWlIMWC4"
    "5ZE4ULkeUH+gHlGBKK5MSBUxDFC0m51D1C4lkFs7QZVXyClUACycM5t5KPPcfNpaZ4lFRdEV7/sp+fr7OjVWzacr"
    "J9QhgqQSXtWdnnVHF3yRdSc4OFyS2VIGh7S6URXxkOsh4V1hK7wsUuB1yfz2IM6hO/UhXL5lCRfw4rfQoo6ET3Kd"
    "9KVZ6qr04up3vn0sSfiBzG3m0rZvoIaEJOHHQQhnPxs88JZc8JubpOLOB5aKB19Bj/zFyydf4+NX+Pd1W0IdxqcH"
    "st5Fuw36cx8v8Bh8QSTlV+QL/sq8w2VBBsBV/vUnTe6QUi++ALav8I+rGnjZQ52qp0ik5Cdka/qlu6+UKB+UJMoH"
    "KGdInOcNixopXuRwQR9BOgewa4vTykcya5+6cMvitCsTyq1sv6bA7n1ZR5fNnju62O6uqanLVk+YQPekPZSSvHfx"
    "hBSZVGDBwaZSt3qpjvrmcBmAC653qy6pSqkXdsVb6ZfluZoQ2ELhWr1DFxseKgRkVReuralCK1UosuyoStQby81W"
    "bwuxhWgpX5EwJnuiLmGqqsml6h2fReeM5UXZxBU4/S4rD6lqlPKvFZd9ML8a/DYJq/oOTapG/FPFaTm5nx3jO8c/"
    "buWcsydp5kr8yWn6EHYMXi47PtTlKV/u50Ew2mzLWIn3f/n3YvX96ntdWq2L77JiqRJxu0B58ZtCIAkf2QqBsAID"
    "jvInXSHZoIBkt8ukNpkd2ebYkkGVLYaWZW8sljjWRWFlF7VvXazfj8o0jPG2JBCyEm2GSeqqhUEsyNZF+9tFStuk"
    "Z9iI2HTP7s08SliifdIz35AhT4e7wTRY63JMe0D4qdKFZeqwVCyTfV+hXlk59DWlz27OFu9AuYINFLoKtnXB6MKh"
    "EaSPmgzytspRZANyp5SL5RYUMK8iFauQZkygqARCPqu4smbd7RUzqYLpjFNFsxUKmWwXiGI3JXtYqbDFMLJSpnaj"
    "Koju/K5qVtcu85epu9ATAOsVsphw436GcjOOe9NcSnFfhz/zFQJlUy4adlOFwENdfG+fHRAKSxjqyiWCV8/gbdwr"
    "9aYcS7aaEaapVvUFIb/AywEx1Fv5MLTfINYoJ+bX/5QPPIXY+l+rCwFmS7qhECA3LAr3fJEKEBl91ZpNXjdJQVnI"
    "Hl0xr9ILkbThZ+Yn39k5gkfnzBsA1uOyZP+mnPwuyQj0l7bSWtNAERamM0dvOIaOK7NR7bZz+yJeu5L73dS3jzXa"
    "SOkshznyEiOP/FoJcfVgU01Hq6t1cJfNy6BX0iFDH07IxZxDOkgsJwFcmvUjkv6Uzrefr6G4r4OUSymwuvLk/fwD"
    "9/UD8Hl+Zspe7FtlHnMdMwRzF5qFjso62bPcQSCNkd6Gm/gIe12mOY2RctJwwlgdS5Xm2BtfiUToKmSm/jDO4P09"
    "UIHyC7FeUVWScTzHOASuBpcbDPh12ys7he/eKQ+EM4L0qUSyZypma+c6sRKE/+MYIjen7C5Q8oCtlen1H9OPq6J1"
    "2uL93/wF62mhwgVfUfVfkns6DT52CWKqI1Us4d521xS7DAP2AMRqlvkqKrOMKspX6aizSJ1aGbvBFS14ldA+KN+f"
    "64Wp0F94C3UZVKqWKPUb9Lhg07R8OejRHrsx9sj6qS/t86V9dbhP/ITj3Eje40CEHaqlrX+oGAdzo2vfgOLbtB77"
    "3HrMekRekCoTwUYtkMYlp8xnPD08zBc6dIHLVLT1pa5ss2/a7Jg28gWWDHmLNqmdcPizgirJ9k0NbEWT9qtpEjcu"
    "EiS+mOMFJgS+tFs9z7hLZWFmq9BHSc53Cty9J/UEWb1M0xCJYrkF0wVeMX3RkS38ZqTNkL4JygrCSiZQAzdV1xpw"
    "dXLBl+oFDJRmQK5Sc0BzJYIXOSZJcVjywue6fC1nW+iINBY56+QE9QKHidSEKNCNDohLNdzzpsB7E5ZO+GWEIyq1"
    "SjYmr1WJK2rJGnNvwhZqV0YXpglXpo48mBfPjLBEFiem/AocYnmed3MAKtYCLFRkGufNnI281OFpayzGkKV5e+wG"
    "of6/W/opC2fTdXT9x5mMB6a9NyZ1eZACjW8EEE2hdBUPWgIlktM9c7rzNyuNkda8dilEgcdiMyOFBSnG7rEqQs4E"
    "OQd1X6FRxVsft/MswXRkXj8rQ8l/+AMM56fXf0elavhdgXD7h0n2jkCrvow+hw/LycA17EcMlyec55Ef8/2//SMG"
    "VaNRyH7wkflkx/BJsO4TWALS63cfwVs/GaBIUkoLTlRdKOj7eMPQQVsxwQud2aVrgjV7Ocdku6MxRFXMz2XB1uem"
    "UF3IZDlD8FbtW4NLIUXqnQ96jrlX1MvRdeBx8+JI+4ReSr/AcQuxby4dVM2tS2+60p3UvPml4tUvHdeCXz67tQhY"
    "xfVttd9q0qB30ZlfjteCwR0GyEK6lamq1suyQq3Hqq0AvcyrqgrKUIgjeXxxKoq10yZ516ql2ENeqUcWKgPNqTxt"
    "lcfThRumXHxlopzT5aa1TAhHm4fg+uCKW79kEhkpN9FbT8UqvDXlYN/aySZUDNYsUKIQv6ghqU3Ule+N0lUYDOZ5"
    "N5gG6ldXSifLZAiqBTq38jpPKPEnaQkO51KBUZGdo5hf/IWsGQq6JANqKEGKY6qK5C6wwhFwprWO+32xu+9Z0e0A"
    "NXxIMglX1kWSs9p2w7aaF3yQSmFWlc1JI+E8JiLDdiJTL184Auy5nL/0FeExU2FJy31k4gER2LW2CCh8QpFyQJQd"
    "Txa91E57cirQe8slqQeGjYLy5jsyE3gU++y90kwkIZmEo8mHoXqPLK36/wJET0cK"
)).decode("utf-8"))
print("✔ Vérifications chargées : vous pouvez commencer.")

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

### Rappels : les méthodes à un pas 📖

On considère le problème de Cauchy

$$y'(t) = f\big(t, y(t)\big), \quad t \in [t_0, T], \qquad y(t_0) = y_0,$$

et la grille $t_i = t_0 + i\,h$, $i = 0, \dots, N$, de pas $h = \dfrac{T - t_0}{N}$.

**Méthode à un pas.** Les méthodes d'Euler et d'Euler modifiée du TP 2 sont de la même forme :

$$\boxed{\; w_0 = y_0, \qquad w_{i+1} = w_i + h\, \varphi(t_i, w_i, h), \quad i = 0, 1, \dots, N - 1, \;}$$

où la **fonction d'incrément** $\varphi$ est une « pente moyenne » sur le pas :

| Méthode | $\varphi(t, w, h)$ |
|:--|:--|
| Euler | $f(t, w)$ |
| Euler modifiée | $\frac{1}{2}\big[f(t, w) + f\big(t + h,\, w + h\,f(t, w)\big)\big]$ |

Pour définir une méthode à un pas, il suffit donc de choisir $\varphi$.

**Erreur de consistance** (ou erreur locale de troncature). C'est ce qu'il manque à la **solution exacte** pour
vérifier le schéma :

$$\tau_{i+1}(h) = \frac{y(t_{i+1}) - y(t_i)}{h} - \varphi\big(t_i, y(t_i), h\big).$$

Pour Euler, on a vu au TP 2 que $\tau_{i+1}(h) = \frac{h}{2}\, y''(\xi_i) = O(h)$. On dit que la méthode est
**consistante d'ordre $p$** si $\tau_{i+1}(h) = O(h^p)$.

**Théorème (de l'erreur locale à l'erreur globale).** On considère la méthode à un pas ci-dessus, avec $\varphi$ continue et $L$-lipschitzienne par rapport à $w$. Notons

$$\tau(h) = \max_{1 \leq i \leq N} |\tau_i(h)|$$

la **plus grande** erreur de consistance sur la grille (elle ne dépend que de $h$). Alors l'**erreur globale**
$e_i = y(t_i) - w_i$ vérifie, pour tout $i$,

$$|e_i| = |y(t_i) - w_i| \leq \frac{\tau(h)}{L}\left(e^{L(t_i - t_0)} - 1\right).$$

*Idée de la preuve* (comme au TP 2) : en soustrayant $y(t_{i+1}) = y(t_i) + h\,\varphi\big(t_i, y(t_i), h\big) + h\,\tau_{i+1}(h)$
et $w_{i+1} = w_i + h\,\varphi(t_i, w_i, h)$, on obtient $|e_{i+1}| \leq (1 + hL)\,|e_i| + h\,\tau(h)$, puis on conclut par
récurrence avec $e_0 = 0$. *(On trouve aussi cette borne écrite avec $e^{L(t_i - t_0)}$ au
lieu de $e^{L(t_i - t_0)} - 1$ : elle est alors un peu moins précise.)*

Pour Euler, $\tau(h) \leq \frac{hM}{2}$ : on retrouve exactement la borne du TP 2. **Erreur de consistance en
$O(h^p)$ $\Longrightarrow$ erreur globale en $O(h^p)$** : pour obtenir une méthode d'ordre $p$, il suffit de trouver
un $\varphi$ dont l'erreur de consistance est en $O(h^p)$.

### Rappels : la méthode de Taylor d'ordre $n$ 📖

**L'idée.** La pente « idéale » serait le taux d'accroissement $\frac{y(t_i + h) - y(t_i)}{h}$ : avec elle, on aurait
$\tau = 0$. On ne le connaît pas, mais la formule de Taylor le développe :

$$\frac{y(t_i + h) - y(t_i)}{h} = y'(t_i) + \frac{h}{2}\, y''(t_i) + \frac{h^2}{6}\, y'''(t_i) + \dots
+ \frac{h^{n-1}}{n!}\, y^{(n)}(t_i) + O(h^n).$$

Euler ne garde que le premier terme, d'où $\tau = O(h)$. En gardant $n$ termes, on obtient $\tau = O(h^n)$.

**Les dérivées de $y$ se calculent avec $f$.** On a $y'(t) = f\big(t, y(t)\big)$ et, en dérivant par rapport à $t$
(dérivée d'une fonction composée) :

$$y''(t) = \frac{d}{dt}\Big[f\big(t, y(t)\big)\Big] = \partial_t f + \partial_y f \cdot y'(t)
= \big(\partial_t f + f\, \partial_y f\big)\big(t, y(t)\big).$$

On note $f' = \partial_t f + f\, \partial_y f$ cette **dérivée totale**, puis, de proche en proche,
$f^{(k)} = \partial_t f^{(k-1)} + f\, \partial_y f^{(k-1)}$. Alors $y^{(k+1)}(t) = f^{(k)}\big(t, y(t)\big)$.

**Méthode de Taylor d'ordre $n$.** C'est la méthode à un pas de fonction d'incrément

$$\boxed{\; T^{(n)}(t, w) = f(t, w) + \frac{h}{2}\, f'(t, w) + \frac{h^2}{6}\, f''(t, w) + \dots
+ \frac{h^{n-1}}{n!}\, f^{(n-1)}(t, w) = \sum_{k=0}^{n-1} \frac{h^k}{(k+1)!}\, f^{(k)}(t, w). \;}$$

La méthode de Taylor d'ordre 1 est la méthode d'Euler.

> ⚠️ **Deux pièges classiques.**
> - $f'$ est la dérivée **totale** $\partial_t f + f\,\partial_y f$, et non la dérivée partielle $\partial_t f$ :
>   $y$ dépend de $t$ !
> - Le coefficient de $f^{(k)}$ est $\dfrac{h^k}{(k+1)!}$, et non $\dfrac{h^k}{k!}$ : pour $k = 1$, c'est $\dfrac{h}{2}$.

**Exemple fil rouge** (celui du TP 2) :

$$y' = y - t^2 + 1, \quad t \in [0, 2], \quad y(0) = 0{,}5, \qquad \text{de solution exacte }
y(t) = (t + 1)^2 - \tfrac{1}{2}\, e^{t}.$$

## 1. Dérivées totales et deux pas à la main

---
#### ✏️ Exercice 1 — Les dérivées totales de l'exemple fil rouge

Pour $f(t, y) = y - t^2 + 1$, on a $\partial_t f = -2t$ et $\partial_y f = 1$. Calculer **à la main** $f'$, $f''$ et
$f'''$ (chacune est la dérivée totale de la précédente), puis compléter les fonctions Python `f1`, `f2` et `f3`.

In [ ]:
def f(t, y):
    """Second membre de l'exemple fil rouge : y' = y - t^2 + 1."""
    return y - t**2 + 1


def f1(t, y):
    """f'(t, y) = ∂f/∂t + f ∂f/∂y."""
    return ...


def f2(t, y):
    """f''(t, y) = ∂f'/∂t + f ∂f'/∂y."""
    return ...


def f3(t, y):
    """f^(3)(t, y) = ∂f''/∂t + f ∂f''/∂y."""
    return ...

In [ ]:
verifier("exo1", f1, f2, f3)

📖 On range la liste $[f, f', f'', f''']$ ; on s'en servira dans tout le TP. Vérifions aussi que
$y''(t) = f'\big(t, y(t)\big)$ avec la solution exacte, pour laquelle $y''(t) = 2 - \frac{1}{2} e^t$ :

In [ ]:
derivees_fr = [f, f1, f2, f3]       # « fr » pour fil rouge


def y_exacte(t):
    """Solution exacte de l'exemple fil rouge."""
    return (t + 1)**2 - 0.5 * np.exp(t)


tt = np.linspace(0, 2, 5)
print("f'(t, y(t)) :", f1(tt, y_exacte(tt)))
print("y''(t)      :", 2 - 0.5 * np.exp(tt))

**Question.** Pourquoi a-t-on $f''' = f''$ ? Que valent alors les dérivées suivantes $f^{(4)}, f^{(5)}, \dots$ ?

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

---
#### ✏️ Exercice 2 — Deux pas de Taylor d'ordre 2 « à la main »

Avec $h = 0{,}2$, calculer $w_1$ puis $w_2$ par la méthode de Taylor d'ordre 2,
$w_{i+1} = w_i + h\left[f(t_i, w_i) + \frac{h}{2}\, f'(t_i, w_i)\right]$, **directement** (sans boucle).

In [ ]:
h = 0.2
t0, w0 = 0.0, 0.5
w1 = ...
t1 = t0 + h
w2 = ...
print(w1, w2)

In [ ]:
verifier("exo2", w1, w2)

## 2. Programmer la méthode de Taylor

On représente la méthode d'ordre $n$ par la **liste** `derivees` $= [f, f', \dots, f^{(n-1)}]$ de ses $n$
fonctions : l'ordre est `n = len(derivees)`. Par exemple, `derivees_fr[:2]` $= [f, f']$ donne l'ordre 2.

---
#### ✏️ Exercice 3 — La fonction d'incrément

Compléter `phi_taylor(derivees, t, w, h)`, qui renvoie
$T^{(n)}(t, w) = \displaystyle\sum_{k=0}^{n-1} \frac{h^k}{(k+1)!}\, f^{(k)}(t, w)$.

*Indications* : $f^{(k)}(t, w)$ s'écrit `derivees[k](t, w)` ; $(k+1)!$ s'écrit `math.factorial(k + 1)`.

In [ ]:
def phi_taylor(derivees, t, w, h):
    """Fonction d'incrément de la méthode de Taylor d'ordre n = len(derivees) :
    T^(n)(t, w) = f(t, w) + h/2 f'(t, w) + ... + h^(n-1)/n! f^(n-1)(t, w),
    où derivees = [f, f', ..., f^(n-1)] est une liste de fonctions de (t, y)."""
    somme = 0
    for k in range(len(derivees)):
        somme = somme + ...      # le terme h^k / (k+1)! f^(k)(t, w)
    return somme

In [ ]:
verifier("exo3", phi_taylor)

📖 Avec une seule fonction, on retrouve la pente d'Euler ; avec $[f, f']$, la pente $(w_1 - w_0)/h = 1{,}65$ de
l'exercice 2 :

In [ ]:
print(phi_taylor([f], 0, 0.5, 0.2), f(0, 0.5))      # ordre 1 : on doit retrouver f
print(phi_taylor([f, f1], 0, 0.5, 0.2))             # ordre 2

---
#### ✏️ Exercice 4 — La fonction `taylor`

Compléter `taylor(derivees, t0, T, y0, N)`, sur le modèle de la fonction `euler` du TP 2 : c'est la méthode à un
pas $w_{i+1} = w_i + h\,\varphi(t_i, w_i, h)$ avec $\varphi = T^{(n)}$.

In [ ]:
def taylor(derivees, t0, T, y0, N):
    """Méthode de Taylor d'ordre n = len(derivees) pour y' = f(t, y), y(t0) = y0, sur [t0, T] avec N pas.
    derivees = [f, f', ..., f^(n-1)]. Renvoie les tableaux t (les t_i) et w (les w_i), de taille N + 1."""
    t = np.linspace(t0, T, N + 1)
    h = (T - t0) / N
    w = np.zeros(N + 1)
    w[0] = ...
    for i in range(N):
        w[i + 1] = ...       # un pas de la méthode, avec la fonction d'incrément phi_taylor
    return t, w

In [ ]:
verifier("exo4", taylor)

📖 Valeurs obtenues avec $N = 10$ ($h = 0{,}2$) pour les ordres 2 et 4, puis comparaison des ordres 1, 2 et 4 :

In [ ]:
t, w2 = taylor(derivees_fr[:2], 0, 2, 0.5, 10)
t, w4 = taylor(derivees_fr, 0, 2, 0.5, 10)
print(" t_i    Taylor 2   erreur       Taylor 4   erreur")
for ti, a, b in zip(t, w2, w4):
    print(f"{ti:4.1f}   {a:.6f}   {abs(y_exacte(ti) - a):.6f}     {b:.6f}   {abs(y_exacte(ti) - b):.6f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
tt = np.linspace(0, 2, 200)
ax1.plot(tt, y_exacte(tt), "k", lw=2, label="solution exacte")
for n, marque in [(1, "o"), (2, "s"), (4, "D")]:
    t, w = taylor(derivees_fr[:n], 0, 2, 0.5, 4)
    ax1.plot(t, w, marque + "-", label=f"Taylor d'ordre {n}")
ax1.set_title("$N = 4$ ($h = 0{,}5$)")
ax1.set_xlabel("$t$")
ax1.legend()
ax1.grid(alpha=0.3)
for n, marque in [(1, "o"), (2, "s"), (3, "^"), (4, "D")]:
    t, w = taylor(derivees_fr[:n], 0, 2, 0.5, 10)
    ax2.semilogy(t[1:], np.abs(y_exacte(t[1:]) - w[1:]), marque + "-", label=f"ordre {n}")
ax2.set_title("erreur $|y(t_i) - w_i|$, $N = 10$ ($h = 0{,}2$)")
ax2.set_xlabel("$t$")
ax2.legend()
ax2.grid(alpha=0.3)
plt.show()

**Question.** Au TP 2, la méthode d'Euler modifiée (elle aussi d'ordre 2) donnait, avec $h = 0{,}2$, une erreur
d'environ $0{,}072$ en $t = 2$. Comparer avec Taylor d'ordre 2. Combien d'évaluations de fonctions chaque méthode
fait-elle par pas ? Quel travail supplémentaire la méthode de Taylor demande-t-elle *avant* de programmer ?

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

## 3. Erreur de consistance et ordre

**Théorème.** Si $y$ est de classe $\mathcal{C}^{n+1}$, l'erreur de consistance de la
méthode de Taylor d'ordre $n$ vaut

$$\tau_{i+1}(h) = \frac{y(t_{i+1}) - y(t_i)}{h} - T^{(n)}\big(t_i, y(t_i)\big)
= \frac{h^n}{(n+1)!}\, f^{(n)}\big(\xi_i, y(\xi_i)\big) = O(h^n), \qquad \xi_i \in \,]t_i, t_{i+1}[.$$

*Preuve* : c'est le reste de la formule de Taylor à l'ordre $n + 1$, avec $y^{(n+1)} = f^{(n)}(\cdot, y)$.

---
#### ✏️ Exercice 5 — Mesurer l'erreur de consistance

Au point $t_0 = 0$, où $y(0) = 0{,}5$ est connu, calculer pour chaque ordre $n = 1, \dots, 4$ et chaque pas $h$ de
`h_tau` l'erreur de consistance

$$|\tau(h)| = \left|\frac{y(t_0 + h) - y(t_0)}{h} - T^{(n)}\big(t_0, y(t_0)\big)\right|.$$

Ranger ces valeurs dans le tableau `tau` (ligne $n - 1$ pour l'ordre $n$, colonne $j$ pour le pas `h_tau[j]`),
puis calculer **sans boucle** le tableau `rapports` des $\tau(h) / \tau(h/2)$.

In [ ]:
h_tau = np.array([0.2, 0.1, 0.05, 0.025, 0.0125])
tau = np.zeros((4, len(h_tau)))          # ligne n - 1 : ordre n ; colonne j : pas h_tau[j]
for n in range(1, 5):
    for j, h in enumerate(h_tau):        # j = 0, 1, ... et h = h_tau[j]
        tau[n - 1, j] = ...              # |tau(h)| pour la méthode d'ordre n (fonctions derivees_fr[:n])
rapports = ...                           # tau(h) / tau(h/2) pour chaque ordre
print(rapports)

In [ ]:
verifier("exo5", tau, rapports)

**Question.** Quelles valeurs attendiez-vous pour les rapports ? D'après le théorème, vers quoi doit tendre
$\tau(h) / h^n$ quand $h \to 0$ ? Le vérifier avec `tau[:, -1] / h_tau[-1]**np.arange(1, 5)`.

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

---
#### ✏️ Exercice 6 — L'ordre global

Comme au TP 2, on mesure l'erreur **globale** $E(h) = \max_i |y(t_i) - w_i|$ pour des pas de plus en plus petits.

1. Pour chaque ordre $n = 1, \dots, 4$ et chaque $N$ de `N_liste`, calculer $E(h)$ et le ranger dans `E[n - 1, j]`
   (la ligne `E[n - 1]` joue le rôle de `E_liste` du TP 2 pour l'ordre $n$).
2. Calculer **sans boucle** le tableau `ordres` des $\log_2\big(E(h) / E(h/2)\big)$ (indice : `np.log2`).
3. Exécuter la fin de la cellule, qui trace les quatre courbes en échelle log-log.

In [ ]:
N_liste = [10, 20, 40, 80, 160, 320]
h_liste = 2 / np.array(N_liste)
E = np.zeros((4, len(N_liste)))          # E[n - 1, j] : erreur maximale de l'ordre n avec N_liste[j] pas
for n in range(1, 5):
    for j, N in enumerate(N_liste):
        t, w = ...
        E[n - 1, j] = ...
ordres = ...
print(ordres)

fig, ax = plt.subplots(figsize=(7, 5))
for n, marque in zip(range(1, 5), ["o", "s", "^", "D"]):
    ax.loglog(h_liste, E[n - 1], marque + "-", label=f"Taylor d'ordre {n}")
ax.set_xlabel("pas $h$")
ax.set_ylabel("erreur maximale")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.show()

In [ ]:
verifier("exo6", E, ordres, ax)

**Question.** En supposant $E(h) \approx C\,h^n$, estimer (avec les valeurs de `E`) le nombre de pas nécessaire pour
obtenir une erreur maximale inférieure à $10^{-6}$ avec l'ordre 1, l'ordre 2 et l'ordre 4.

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

## 4. Calculer les dérivées avec SymPy

Pour $f(t, y) = y - t^2 + 1$, les dérivées totales se calculent en une ligne. Mais pour
$f(t, y) = 1 + t \sin(t y)$, calculer $f''$ et $f'''$ à la main est long, et une erreur de calcul est vite arrivée.
**SymPy** (vu au TP 1) sait dériver : la dérivée totale $\partial_t g + f\,\partial_y g$ s'écrit
`sp.diff(g, ts) + f_expr * sp.diff(g, ys)`, et `sp.lambdify` transforme une expression en fonction Python 📖.

> On nomme les symboles `ts` et `ys` (« s » pour symbolique), pour ne pas écraser les tableaux NumPy `t` et `w`.

In [ ]:
import sympy as sp

ts, ys = sp.symbols("t y")
f_expr = ys - ts**2 + 1
f1_expr = sp.diff(f_expr, ts) + f_expr * sp.diff(f_expr, ys)     # dérivée totale
display(f1_expr)

f1_num = sp.lambdify((ts, ys), f1_expr, "numpy")      # expression SymPy -> fonction Python de (t, y)
print(f1_num(0.0, 0.5), f1(0.0, 0.5))                  # même valeur que la fonction f1 de l'exercice 1

---
#### ✏️ Exercice 7 — La liste des dérivées totales

Compléter `derivees_totales(f_expr, n)`, qui renvoie la liste des $n$ expressions $[f, f', \dots, f^{(n-1)}]$ :
chaque élément est la dérivée totale du précédent. La fonction `en_fonctions` (fournie) transforme ensuite cette liste
en liste de fonctions Python, que l'on peut donner à `taylor`.

In [ ]:
def derivees_totales(f_expr, n):
    """Liste des n expressions SymPy [f, f', ..., f^(n-1)] (dérivées totales) pour y' = f(t, y)."""
    liste = [f_expr]
    g = f_expr
    for k in range(1, n):
        g = ...              # dérivée totale de g : ∂g/∂t + f ∂g/∂y
        liste.append(g)
    return liste


def en_fonctions(liste):
    """Transforme une liste d'expressions SymPy en liste de fonctions Python de (t, y)."""
    return [sp.lambdify((ts, ys), g, "numpy") for g in liste]

In [ ]:
verifier("exo7", derivees_totales)

📖 Sur l'exemple fil rouge, on retrouve les dérivées de l'exercice 1, et les mêmes valeurs qu'avec les dérivées
écrites à la main :

In [ ]:
for g in derivees_totales(ys - ts**2 + 1, 4):
    display(sp.expand(g))

t, w = taylor(en_fonctions(derivees_totales(ys - ts**2 + 1, 4)), 0, 2, 0.5, 10)
print("w_10 avec SymPy          :", w[-1])
print("w_10 avec f1, f2, f3 (exo 1) :", taylor(derivees_fr, 0, 2, 0.5, 10)[1][-1])

---
#### ✏️ Exercice 8 — Un second membre plus compliqué

On considère

$$y' = 1 + t \sin(t y), \quad t \in [0, 2], \quad y(0) = 0.$$

On ne connaît pas de solution explicite : la solution de **référence** `y_ref` est calculée par `solve_ivp` avec une
très grande précision. Avec $N = 20$ ($h = 0{,}1$) :

1. obtenir avec SymPy la liste `derivees` $= [f, f', f'', f''']$ (fonctions Python) ;
2. calculer les solutions de Taylor d'ordre 2 (`w2`) et d'ordre 4 (`w4`) ;
3. calculer leurs erreurs maximales `E2` et `E4` par rapport à `y_ref`.

In [ ]:
from scipy.integrate import solve_ivp

f_expr = 1 + ts * sp.sin(ts * ys)
N = 20
t_ref = np.linspace(0, 2, N + 1)
sol = solve_ivp(lambda t, y: 1 + t * np.sin(t * y), (0, 2), [0.0], t_eval=t_ref, rtol=1e-12, atol=1e-12)
y_ref = sol.y[0]                 # solution de référence aux instants t_i

derivees = ...                   # [f, f', f'', f'''] : fonctions Python obtenues avec SymPy
t, w2 = ...                      # Taylor d'ordre 2
t, w4 = ...                      # Taylor d'ordre 4
E2 = ...                         # max_i |w_i - y_ref(t_i)| pour l'ordre 2
E4 = ...                         # idem pour l'ordre 4
print(E2, E4)

In [ ]:
verifier("exo8", w2, w4, E2, E4)

📖 **Le prix à payer.** Regardons la taille des dérivées totales de $f(t, y) = 1 + t\sin(ty)$ :

In [ ]:
for k, g in enumerate(derivees_totales(f_expr, 6)):
    print(f"f^({k}) : {sp.count_ops(g):6d} opérations")

print("\nf'' =")
display(derivees_totales(f_expr, 3)[2])

**Question.** D'après ce que vous venez de voir, pourquoi les méthodes de Taylor sont-elles rarement utilisées en
pratique, malgré leur ordre élevé ?

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

## 5. Application : un projectile freiné par l'air

Un projectile de masse $m = 0{,}11$ kg est lancé verticalement vers le haut avec la vitesse $v(0) = 8$ m/s.
Il est soumis à son poids $-mg$ et à la résistance de l'air $-k\,v|v|$, avec
$g = 9{,}8$ m/s² et $k = 0{,}002$ kg/m. Sa vitesse $v(t)$ vérifie donc

$$m\, v' = -m g - k\, v |v| \qquad \Longleftrightarrow \qquad v' = f(t, v) = -g - \frac{k}{m}\, v |v|.$$

> La valeur absolue fait que la force de frottement s'oppose toujours au mouvement : vers le bas quand le projectile
> monte ($v > 0$), vers le haut quand il redescend ($v < 0$).

**Questions.** Calculer la vitesse aux instants $0{,}1$ ; $0{,}2$ ; … ; $1{,}0$ s. Au dixième de seconde
près, quand le projectile atteint-il sa hauteur maximale ?

---
#### ✏️ Exercice 9

1. Calculer à la main la dérivée totale $f'(t, v) = \partial_t f + f\,\partial_v f$ et compléter `f1_proj`.
   *Indication* : la dérivée de $v \mapsto v|v|$ est $v \mapsto 2|v|$ (pour tout $v$, y compris $v < 0$).
2. Appliquer la méthode de Taylor d'ordre 2 avec $h = 0{,}1$ sur $[0, 1]$ : tableaux `t` et `w`, avec
   $w_i \approx v(t_i)$.
3. En déduire `t_max`, le premier instant $t_i$ où $w_i \leq 0$ (avec un masque, comme au TP 2).
4. Plus précisément : si $w_{j-1} > 0 \geq w_j$, estimer l'instant `t_etoile` où $v$ s'annule par interpolation
   linéaire entre $(t_{j-1}, w_{j-1})$ et $(t_j, w_j)$ :
   $t^* \approx t_{j-1} + h\, \dfrac{w_{j-1}}{w_{j-1} - w_j}$. *Indice* : `j = np.argmax(w <= 0)` donne l'indice du
   premier $w_i \leq 0$.

In [ ]:
m, g, k = 0.11, 9.8, 0.002
v0 = 8.0


def f_proj(t, v):
    """Second membre : v' = -g - (k/m) v|v|."""
    return -g - k / m * v * abs(v)


def f1_proj(t, v):
    """Dérivée totale f'(t, v) = ∂f/∂t + f ∂f/∂v."""
    return ...


t, w = ...                       # Taylor d'ordre 2, h = 0.1 sur [0, 1]
for ti, wi in zip(t, w):
    print(f"t = {ti:.1f} s    v ≈ {wi:8.4f} m/s")

t_max = ...                      # premier instant t_i tel que w_i <= 0
j = np.argmax(w <= 0)            # son indice
t_etoile = ...                   # interpolation linéaire entre (t_{j-1}, w_{j-1}) et (t_j, w_j)
print("t_max =", t_max, "   t_etoile =", t_etoile)

In [ ]:
verifier("exo9", f1_proj, w, t_max, t_etoile)

📖 Ce problème a une solution exacte. Avec $a = \sqrt{mg/k}$, $b = \sqrt{gk/m}$ et $\theta_0 = \arctan(v_0/a)$ :
$v(t) = a \tan(\theta_0 - b t)$ tant que le projectile monte, c'est-à-dire jusqu'à $t^* = \theta_0 / b$, puis
$v(t) = -a \tanh\big(b (t - t^*)\big)$ quand il redescend. Comparons :

In [ ]:
a, b = np.sqrt(m * g / k), np.sqrt(g * k / m)
theta0 = np.arctan(v0 / a)
t_etoile_exact = theta0 / b


def v_exacte(t):
    return np.where(t <= t_etoile_exact, a * np.tan(theta0 - b * t), -a * np.tanh(b * (t - t_etoile_exact)))


print(f"t* exact = {t_etoile_exact:.4f} s ;  sans frottement : v0 / g = {v0 / g:.4f} s")
print("erreur maximale de Taylor d'ordre 2 :", np.max(np.abs(w - v_exacte(t))))

tt = np.linspace(0, 1, 200)
t1, w1 = taylor([f_proj], 0, 1, v0, 10)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(tt, v_exacte(tt), "k", lw=2, label="solution exacte")
ax.plot(t1, w1, "o-", label="Euler, h = 0.1")
ax.plot(t, w, "s-", label="Taylor d'ordre 2, h = 0.1")
ax.plot(tt, v0 - g * tt, "k:", label="sans frottement")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("$t$ (s)")
ax.set_ylabel("$v$ (m/s)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

**Question.** Pourquoi le projectile atteint-il sa hauteur maximale plus tôt qu'en l'absence de frottement ? La
fonction $f$ est-elle assez régulière pour appliquer une méthode de Taylor d'ordre 3 ou plus sur tout $[0, 1]$ ?

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

## 6. ★ Entre les points de la grille : l'interpolation d'Hermite

La méthode ne donne des valeurs qu'aux instants $t_i$. Pour estimer $y$ **entre** deux instants, par exemple en
$t = 1{,}25$ (entre $t_6 = 1{,}2$ et $t_7 = 1{,}4$ avec $h = 0{,}2$), on peut interpoler linéairement… mais l'erreur
d'interpolation linéaire est en $O(h^2)$ : elle gâche la précision d'une méthode d'ordre 4 !

Or on connaît aussi les **pentes** $p_i = f(t_i, w_i) \approx y'(t_i)$. Le polynôme d'**interpolation d'Hermite
cubique** sur $[t_i, t_{i+1}]$ prend les valeurs $w_i, w_{i+1}$ et les pentes $p_i, p_{i+1}$ aux extrémités.
Avec $s = \dfrac{t - t_i}{h} \in [0, 1]$ :

$$H(t) = (2s^3 - 3s^2 + 1)\, w_i + (s^3 - 2s^2 + s)\, h\, p_i + (-2s^3 + 3s^2)\, w_{i+1} + (s^3 - s^2)\, h\, p_{i+1}.$$

---
#### ✏️ Exercice 10 ★

Avec la méthode de Taylor d'ordre 4 et $N = 10$, calculer une valeur approchée de $y(1{,}25)$ :

1. par interpolation linéaire (`y_lin`, avec `np.interp(1.25, t, w4)`) ;
2. par interpolation d'Hermite (`y_herm`).

Comparer à la valeur exacte (on doit trouver des erreurs d'environ $7{,}5 \cdot 10^{-4}$ et $2{,}9 \cdot 10^{-5}$).

In [ ]:
t, w4 = taylor(derivees_fr, 0, 2, 0.5, 10)
h = 0.2
y_lin = ...
i = 6                            # 1.25 est entre t_6 = 1.2 et t_7 = 1.4
s = ...
p_i = ...                        # pente f(t_i, w_i)
p_ip1 = ...                      # pente f(t_{i+1}, w_{i+1})
y_herm = ...
print(f"linéaire : {y_lin:.7f} (erreur {abs(y_lin - y_exacte(1.25)):.1e})")
print(f"Hermite  : {y_herm:.7f} (erreur {abs(y_herm - y_exacte(1.25)):.1e})")

In [ ]:
verifier("exo10", y_lin, y_herm)

📖 Sur tout l'intervalle : erreur des deux interpolations entre les points de la grille.

In [ ]:
def hermite(tt, t, w, p):
    # interpolation d'Hermite cubique par morceaux, aux instants tt
    i = np.clip(np.searchsorted(t, tt) - 1, 0, len(t) - 2)
    h = t[i + 1] - t[i]
    s = (tt - t[i]) / h
    return ((2 * s**3 - 3 * s**2 + 1) * w[i] + (s**3 - 2 * s**2 + s) * h * p[i]
            + (-2 * s**3 + 3 * s**2) * w[i + 1] + (s**3 - s**2) * h * p[i + 1])


tt = np.linspace(0, 2, 801)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(tt, np.abs(np.interp(tt, t, w4) - y_exacte(tt)), label="interpolation linéaire")
ax.semilogy(tt, np.abs(hermite(tt, t, w4, f(t, w4)) - y_exacte(tt)), label="interpolation d'Hermite")
ax.semilogy(t, np.abs(w4 - y_exacte(t)), "ko", ms=4, label="erreur aux points $t_i$")
ax.set_xlabel("$t$")
ax.set_title("Taylor d'ordre 4, $h = 0{,}2$ : erreur entre les points de la grille")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.show()

## 7. ★ Stabilité

Au TP 2, pour $y' = -10\,y$, la méthode d'Euler n'était stable que pour $h < 0{,}2$. Qu'en est-il des méthodes de
Taylor ? Pour $y' = \lambda y$, on a $f(t, y) = \lambda y$, puis $f' = \partial_t f + f\,\partial_y f = \lambda^2 y$
et plus généralement $f^{(k)}(t, y) = \lambda^{k+1} y$. Le schéma devient

$$w_{i+1} = w_i + h \sum_{k=0}^{n-1} \frac{h^k \lambda^{k+1}}{(k+1)!}\, w_i = R_n(h\lambda)\, w_i, \qquad
R_n(z) = 1 + z + \frac{z^2}{2} + \dots + \frac{z^n}{n!},$$

le polynôme de Taylor de $e^z$ ! Donc $w_i = R_n(h\lambda)^i\, y_0 \to 0$ si et seulement si $|R_n(h\lambda)| < 1$.

---
#### ✏️ Exercice 11 ★

Pour $\lambda = -10$ et $n = 1, 2, 3, 4$, calculer le pas critique : le plus grand $h$ tel que $|R_n(-10 h)| \leq 1$.
On pourra chercher, sur une grille fine de $x = h\lambda \in [-4, 0]$, le plus petit $x$ tel que $|R_n(x)| \leq 1$.
Ranger les quatre valeurs dans la liste `h_critiques`.

In [ ]:
x = np.linspace(-4, 0, 400001)          # x = h * lambda
h_critiques = []
for n in range(1, 5):
    R = ...                             # R_n(x) = somme des x^k / k! pour k = 0, ..., n
    x_min = ...                         # plus petit x tel que |R_n(x)| <= 1
    h_critiques.append(-x_min / 10)
print(h_critiques)

In [ ]:
verifier("exo11", h_critiques)

In [ ]:
x = np.linspace(-3.5, 0.5, 400)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhspan(-1, 1, color="green", alpha=0.08, label="zone stable $|R_n| \\leq 1$")
for n in range(1, 5):
    ax.plot(x, sum(x**k / math.factorial(k) for k in range(n + 1)), label=f"$R_{n}$")
ax.plot(x, np.exp(x), "k--", label="$e^x$")
ax.set_ylim(-2, 2)
ax.set_xlabel(r"$x = h\lambda$")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

**Question.** Monter en ordre améliore-t-il beaucoup la stabilité ? Que faudrait-il pour une équation très
« raide » comme $y' = -1000\,y$ ?

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

## 8. Aide-mémoire

| Besoin | Commande |
|:--|:--|
| Factorielle | `math.factorial(k)` |
| Liste des $n$ premières fonctions | `derivees[:n]` |
| Parcourir un tableau avec ses indices | `for j, h in enumerate(h_liste):` |
| Rapports de valeurs successives, ordre | `E[:, :-1] / E[:, 1:]`, `np.log2(...)` |
| Symboles, dérivée partielle | `ts, ys = sp.symbols("t y")`, `sp.diff(g, ts)` |
| Dérivée totale | `sp.diff(g, ts) + f_expr * sp.diff(g, ys)` |
| Expression → fonction Python | `sp.lambdify((ts, ys), g, "numpy")` |
| Taille d'une expression | `sp.count_ops(g)` |
| Interpolation linéaire | `np.interp(t_voulu, t, w)` |
| Indice du premier `True` | `np.argmax(w <= 0)` |

### À retenir

- **Méthode à un pas** : $w_{i+1} = w_i + h\,\varphi(t_i, w_i, h)$. Erreur de consistance
  $\tau(h) = \frac{y(t_{i+1}) - y(t_i)}{h} - \varphi$ en $O(h^p)$ et $\varphi$ lipschitzienne
  $\Longrightarrow$ erreur globale en $O(h^p)$.
- **Taylor d'ordre $n$** : $\varphi = T^{(n)} = \sum_{k=0}^{n-1} \frac{h^k}{(k+1)!}\, f^{(k)}$, avec les dérivées
  **totales** $f^{(k)} = \partial_t f^{(k-1)} + f\,\partial_y f^{(k-1)}$. Méthode **d'ordre $n$** ; Euler = ordre 1.
- **Inconvénient** : il faut calculer les dérivées de $f$ (à la main ou avec SymPy), et leur taille explose.

### Et au prochain TP…

Les **méthodes de Runge–Kutta** atteignent le même ordre que Taylor **sans dériver $f$** : elles combinent plusieurs
évaluations de $f$ en des points bien choisis (Euler modifiée en était déjà un exemple, d'ordre 2).

**N'oubliez pas de télécharger votre notebook (File → Download) avant de partir !**